<a href="https://colab.research.google.com/github/Toan-ten/Arxiv-Project-Group10/blob/feat%2Fm1-pipeline-optimization/notebooks/m1_pipeline_thu_thap_va_kiem_toan.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Mở trong Colab"/></a>

# Quy trình thu thập và kiểm toán M1

## Mục tiêu và phạm vi

Sổ tay vận hành quy trình M1 trên danh sách ID được giao cho một thành viên. Đơn vị đầu vào là một vị trí ID arXiv trong cấu hình theo MSSV; vị trí đó có thể không tồn tại, có nhiều phiên bản hoặc chưa có đủ bằng chứng. Mỗi thành viên có 5.000 vị trí, chia thành 50 lô, mỗi lô 100 ID. Bốn dải tạo thành phạm vi 20.000 vị trí của nhóm. Quy trình thu thập toàn bộ các dải được giao, không lấy mẫu ngẫu nhiên từ toàn kho arXiv.

## Thiết kế thu thập

arXiv là nguồn xác nhận bài và danh sách phiên bản. Pha này truy vấn Atom theo lô tối đa 20 ID. Bản chụp Kaggle là nguồn tùy chọn để lập chỉ mục ứng viên và đối chiếu danh sách phiên bản cùng ngày cập nhật; nếu bản chụp thiếu hoặc lệch, quy trình ưu tiên kết quả arXiv. Mã nguồn của từng phiên bản vẫn được tải riêng từ endpoint arXiv có mã phiên bản. Kaggle không thay thế việc xác nhận arXiv hoặc tải mã nguồn.

S2 lấy siêu dữ liệu theo lô tối đa 50 ID trong cấu hình M1 và tải tài liệu tham khảo riêng cho từng bài.

Sổ tay có ba pha:

1. arXiv xác minh từng ID, lập danh sách phiên bản và thu thập mã nguồn.
2. Semantic Scholar truy vấn siêu dữ liệu và tài liệu tham khảo cho cùng danh sách ID, độc lập với tiến độ arXiv.
3. Kiểm toán xác minh dấu vân tay phạm vi, xuất xứ, định danh, mã kiểm tra và trạng thái phân trang trước khi hợp nhất dữ liệu và xuất bảng.

Mỗi pha có điểm khôi phục riêng. Mặc định chạy arXiv → S2 → kiểm toán; có thể chạy S2 trước hoặc chạy lại riêng một pha. Trạng thái S2 `not_found` không xác nhận ID arXiv không tồn tại.

## Tái lập và giới hạn kết luận

Môi trường chạy lấy mã theo SHA đầy đủ được ghim trong ô cấu hình. Ô này in dấu vân tay phạm vi, dải ID và thư mục lưu trữ; hãy giữ các thông tin đó cùng nhật ký phiên. `COHORT_LIMIT = None` chạy đủ 5.000 vị trí; `COHORT_LIMIT = 1` chạy thử 100 vị trí trong thư mục riêng. Lượt chạy thử chỉ kiểm tra quy trình vận hành, không đại diện cho phép đo toàn nhóm.

Sổ tay không chứa kết quả đo hiệu năng. Chỉ dùng số liệu từ báo cáo kiểm toán và phép đo đã tổng hợp cho phạm vi đầy đủ. Thời gian thử lại và chờ API được tính; khoảng nghỉ giữa các phiên được loại trừ. Xem [hướng dẫn chạy trên Colab](../docs/M1_COLAB_MEMBER_GUIDE.md), [hợp đồng dữ liệu nguồn](../docs/contracts/source_data_contracts.md), [arXiv API](https://info.arxiv.org/help/api/user-manual.html) và [Semantic Scholar API](https://api.semanticscholar.org/api-docs/graphs).

## 1. Cấu hình môi trường và phạm vi

In [ ]:
# Pin notebook tới commit runtime chứa mã M1 đã cập nhật.
import os
from pathlib import Path
import re
import subprocess
import sys

M1_RUNTIME_COMMIT = "3326330c38b8e69b99110493c1f25e9869632b9d"
MSSV = ""  # Nhập MSSV trong config/group10_id_ranges.json.
COHORT_LIMIT = None  # None: 50 cohort; 1..50: số cohort đầu tiên.
PILOT_ROOT = COHORT_LIMIT is not None
DRIVE_BASE = Path("/content/drive/MyDrive/M1-pipeline-optimization-pilot" if PILOT_ROOT else "/content/drive/MyDrive/M1-pipeline-optimization-full")
LOCAL_BASE = Path("/content/m1-pipeline-optimization-pilot-local" if PILOT_ROOT else "/content/m1-pipeline-optimization-local")

loaded_src = sorted(name for name in sys.modules if name == "src" or name.startswith("src."))
if loaded_src:
    raise RuntimeError("Kernel đã nạp src trước khi checkout. Chọn Runtime > Restart session rồi chạy lại cell này.")
if os.environ.get("COLAB_GPU") == "1" or os.environ.get("COLAB_TPU_ADDR"):
    raise RuntimeError("Chọn runtime chỉ dùng CPU trong Runtime settings.")
if not MSSV.strip():
    raise ValueError("Nhập MSSV trước khi tiếp tục.")

from google.colab import drive
drive.mount("/content/drive")

REPO_DIR = Path("/content/Arxiv-Project-Group10")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "https://github.com/Toan-ten/Arxiv-Project-Group10.git", str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
if not re.fullmatch(r"[0-9a-f]{40}", M1_RUNTIME_COMMIT):
    raise RuntimeError("M1_RUNTIME_COMMIT phải có đủ 40 ký tự SHA.")
subprocess.run(["git", "fetch", "origin"], check=True)
subprocess.run(["git", "checkout", "--detach", M1_RUNTIME_COMMIT], check=True)
current_head = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
if current_head != M1_RUNTIME_COMMIT:
    raise RuntimeError("Mã runtime không khớp SHA đã ghim.")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

from src.m1_workflow import build_member_selection
SELECTION = build_member_selection(MSSV, COHORT_LIMIT)
DRIVE_ROOT = DRIVE_BASE / MSSV
LOCAL_ROOT = LOCAL_BASE / MSSV
print(f"Runtime: {current_head}")
print(f"MSSV: {MSSV}; IDs: {SELECTION['expected_count']}; cohorts: {len(SELECTION['cohorts'])}")
print(f"Selection fingerprint: {SELECTION['fingerprint']}")
print(f"Drive root: {DRIVE_ROOT}; local root: {LOCAL_ROOT}")


## 2. Pha arXiv: xác minh và thu thập mọi phiên bản

Pha này chỉ ghi bằng chứng arXiv và mã nguồn; pha này không chứng nhận M1 đã hoàn tất. Chạy ô để xác minh danh sách ID, xử lý các phiên bản được phát hiện và lưu điểm khôi phục. Khi truy vấn discovery dừng vì HTTP 429, ô có thể tự tiếp tục tối đa một lần mỗi lô và hai lần mỗi phiên, sau khoảng chờ ngẫu nhiên 5–10 phút. Chỉ tiếp tục khi bản ghi lượt gọi, điểm khôi phục và kho lưu trữ đã được xác minh. HTTP 403, HTTP 429 ở pha khác, lỗi điểm khôi phục hoặc kiểm toán, hết ngân sách và các lỗi khác vẫn làm ô dừng. Khoảng chờ được tính vào thời gian chạy và phép đo. Nếu dừng ô, hãy chạy lại với cùng MSSV, phạm vi, SHA môi trường chạy và thư mục gốc Drive.

In [ ]:
from src.m1_workflow import run_m1_arxiv_member
try:
    from google.colab import userdata
    KAGGLE_API_TOKEN = userdata.get("KAGGLE_API_TOKEN")
except Exception:
    KAGGLE_API_TOKEN = None


ARXIV_RESULT = run_m1_arxiv_member(
    MSSV,
    drive_root=DRIVE_ROOT,
    local_root=LOCAL_ROOT,
    cohort_limit=COHORT_LIMIT,
    kaggle_token=KAGGLE_API_TOKEN,
    auto_resume_429=True,
    auto_resume_429_max_per_cohort=1,
    auto_resume_429_max_per_session=2,
    auto_resume_429_cooldown_min_seconds=300,
    auto_resume_429_cooldown_max_seconds=600,
)
print({key: ARXIV_RESULT.get(key) for key in (
    "stage", "selection_fingerprint", "expected_id_count",
    "source_complete_papers", "stopped", "stop_reason", "m1_complete", "kaggle", "auto_resume_429",
)})


## 3. Pha Semantic Scholar: siêu dữ liệu và tài liệu tham khảo

S2 truy vấn cùng danh sách ID, không phụ thuộc việc pha arXiv đã chạy hay chưa. Điểm khôi phục lưu siêu dữ liệu, tài liệu tham khảo thô, trạng thái phân trang, vị trí tiếp tục, lỗi và mã kiểm tra. Khi các lượt thử nội bộ đã hết và gặp HTTP 429 ở siêu dữ liệu hoặc tài liệu tham khảo, ô có thể tự tiếp tục tối đa một lần mỗi lô và hai lần mỗi phiên sau khoảng chờ ngẫu nhiên 5–10 phút. Chỉ tiếp tục khi điểm khôi phục đúng dấu vân tay phạm vi đã được xác minh. Nếu hết ngân sách, điểm khôi phục sai, gặp HTTP 403 hoặc vượt giới hạn, ô dừng để người vận hành tiếp tục thủ công khi phù hợp. Hệ thống tôn trọng `Retry-After` còn hiệu lực; thời gian chờ được tính vào thời gian chạy và phép đo.

Thêm `SEMANTIC_SCHOLAR_API_KEY` vào Colab Secrets nếu được cấp khóa và bật quyền truy cập notebook. Không dán khóa vào mã nguồn hoặc nhật ký. Nếu không có khóa, API có thể áp dụng hạn mức khác. S2 `not_found` là trạng thái tra cứu của Semantic Scholar, không thay đổi kết luận arXiv.

In [ ]:
from google.colab import userdata
from src.m1_workflow import run_m1_s2_member

try:
    S2_API_KEY = userdata.get("SEMANTIC_SCHOLAR_API_KEY")
except Exception as exc:
    print(f"Không đọc được Colab Secret ({type(exc).__name__}); lượt S2 dùng quota không xác thực.")
    S2_API_KEY = None

S2_RESULT = run_m1_s2_member(
    MSSV,
    drive_root=DRIVE_ROOT,
    local_root=LOCAL_ROOT,
    semantic_scholar_api_key=S2_API_KEY,
    cohort_limit=COHORT_LIMIT,
    auto_resume_429=True,
    auto_resume_429_max_per_cohort=1,
    auto_resume_429_max_per_session=2,
    auto_resume_429_cooldown_min_seconds=300,
    auto_resume_429_cooldown_max_seconds=600,
)
S2_API_KEY = None
S2_AUTO_RESUME = S2_RESULT.get("auto_resume_429", {})
S2_RESULT_SUMMARY = {key: S2_RESULT.get(key) for key in (
    "stage", "selection_fingerprint", "expected_id_count",
    "metadata_complete", "references_complete", "stopped", "stop_reason", "m1_complete",
)}
S2_RESULT_SUMMARY["auto_resume_429"] = {key: S2_AUTO_RESUME.get(key) for key in (
    "resume_attempts", "cooldown_seconds", "events_path", "events_sha256",
)}
print(S2_RESULT_SUMMARY)


## 4. Kiểm toán, đối soát và xuất bảng

Chạy ô sau khi arXiv và S2 đã ghi đủ bằng chứng. Kiểm toán đối chiếu dấu vân tay phạm vi, định danh, xuất xứ, mã kiểm tra và độ phủ phân trang. Chỉ hợp nhất dữ liệu S2 khi bằng chứng hợp lệ, và chỉ xuất bảng khi toàn bộ phạm vi đạt điều kiện M1. Nếu kiểm toán bị chặn, xem lỗi và trạng thái trong báo cáo; không thay dữ liệu thiếu bằng giá trị rỗng hoặc 0.

In [ ]:
from src.m1_workflow import run_m1_audit_member

AUDIT_RESULT = run_m1_audit_member(
    MSSV,
    drive_root=DRIVE_ROOT,
    local_root=LOCAL_ROOT,
    cohort_limit=COHORT_LIMIT,
)
print({key: AUDIT_RESULT.get(key) for key in (
    "selection_fingerprint", "expected_id_count", "merged_s2_ids",
    "is_reconciled", "integrity_failed_papers", "unprocessed_papers",
    "export_status", "export_error", "report_path", "benchmark_summary_path",
)})


## Tiếp tục và đo hiệu năng

Nếu chạy S2 trước arXiv, chạy ô arXiv rồi chạy lại kiểm toán. Khi tiếp tục, dùng cùng MSSV, phạm vi, SHA môi trường chạy và thư mục gốc Drive. Điểm khôi phục không tương thích hoặc sai mã kiểm tra được giữ để kiểm tra; không xóa điểm khôi phục và không chạy hai môi trường cùng ghi một phạm vi.

Lượt chạy thử 100 ID chỉ dùng để kiểm tra khả năng dừng, tiếp tục và kiểm toán. Phép đo cuối phải bao phủ đủ 20.000 vị trí ID trên CPU Colab; giữ nhật ký từng phiên, cộng thời gian thử lại và chờ API, đồng thời loại khoảng nghỉ giữa các phiên. Không suy rộng thời gian, RAM hoặc dung lượng từ lượt chạy thử. Xem [quy trình M1](../docs/operations/m1_production.md) và [hướng dẫn tổng hợp benchmark](../docs/operations/benchmark_runner.md).